# Fast Hyperparameter Parameter Tuning /w XGBoost & Optuna

5-15x Faster Tuning with GPU Accelerated XGBoost 3.0!

- XGBoost is one of the most powerful algorithms for modeling tabular data.
- Hyperparameter tuning allows us to squeeze extra performance out of these models and improve their accuracy for the dataset type.
- Tuning takes a long time - but with XGBoost's GPU support we can now speed up the process substantially! 

## Some Notes to keep in mind before we start:

1. Hyperparameter tuning is WORTHLESS without a proper validation setup.
1. Avoid the endless hyperparameter tuning!
   - Rule of thumb: tune once to get baseline params.
   - Then tune again at the end of feature engineering to get an added boost.
3. Tuning paradox:
   - Bad parameters can SIGNIFICANTLY hurt your model.
   - At the same time the difference between 'okay' and 'perfect' parameters is usually minimal.
   - Small improvements can still be a big deal in production systems.

# Setup

In [ ]:
# Lets upgrade to the latest version of XGboost (3.0)
!pip install --upgrade -q xgboost

In [ ]:
# For code formatting
!pip install -q jupyter-black "black>=24.0.0" ruff nbqa
%load_ext jupyter_black

In [ ]:
import pandas as pd
import xgboost as xgb
import numpy as np
import sklearn
import optuna

print("XGBoost Version", xgb.__version__)
print("Pandas Version", pd.__version__)
print("Numpy Version", np.__version__)
print("Sklearn Version", sklearn.__version__)
print("Optuna Version", optuna.__version__)

In [ ]:
# Confirm we have a GPU available in the instance
!nvidia-smi

# Why Hyperparameter Tuning?

- A bunch of knobs to turn. Which is best?
- Each dataset might require different parameters to have the best performance.
- Parameters don't work in isolation - we don't only need to find the best parameter individually - but we need to find the best **combination** of paramters for our dataset.

Below is an example of all the parameters avaiable for a classification model.

In [ ]:
# Here we can wee all the possible parameters that
reg = xgb.XGBClassifier(
    base_score=None,
    booster=None,
    callbacks=None,
    colsample_bylevel=None,
    colsample_bynode=None,
    colsample_bytree=None,
    device=None,
    early_stopping_rounds=None,
    enable_categorical=False,
    eval_metric=None,
    feature_types=None,
    gamma=None,
    grow_policy=None,
    importance_type=None,
    interaction_constraints=None,
    learning_rate=None,
    max_bin=None,
    max_cat_threshold=None,
    max_cat_to_onehot=None,
    max_delta_step=None,
    max_depth=None,
    max_leaves=None,
    min_child_weight=None,
    monotone_constraints=None,
    multi_strategy=None,
    n_estimators=None,
    n_jobs=None,
    num_parallel_tree=None,
    random_state=None,
)

# Key XGBoost Parameters

### **`eta` (a.k.a `learning_rate`)**
- **What it does:** Controls how big a step the model takes when updating weights after each round of boosting.  
- **Why it matters:**  
  - A **small value** = slower learning but more stable and accurate.  
  - A **large value** = faster learning but risks skipping over important patterns.  
- **Think of it as…** the speed of your car. Go too fast (high learning rate) and you might miss turns. Go slower (low learning rate) and you’ll reach the destination more carefully.

### **`max_depth`**
- **What it does:** Sets how deep each decision tree can grow.  
- **Why it matters:**  
  - **Deeper trees** capture more detail but can overfit (memorize the training set).  
  - **Shallow trees** are simpler and generalize better but may miss complexity.  
- **Think of it as…** how many follow-up questions you allow when making a decision. More questions (deeper tree) = more detail, but maybe too much detail.

### **`n_estimators`**
- **What it does:** The number of boosting rounds (i.e., how many trees are built in total).  
- **Why it matters:**  
  - **More trees** can improve accuracy, but also make the model slower and prone to overfitting if unchecked.  
- **Think of it as…** the number of attempts you give yourself to refine a guess. More tries can help, but too many might just repeat the same mistakes.

### **`subsample`**
- **What it does:** The fraction of the training data used to build each tree.  
- **Why it matters:**  
  - Using **less than 1.0** (e.g., 0.8) helps prevent overfitting by adding randomness.  
- **Think of it as…** asking only a portion of your friends for advice instead of everyone. It keeps things varied and avoids relying on the same crowd every time.

### **`colsample_bytree`**
- **What it does:** The fraction of features (columns) considered when building each tree.  
- **Why it matters:**  
  - Forces the model to try different combinations of features, making it more robust.  
- **Think of it as…** trying different ingredients each time you cook. You don’t always need the full spice rack to make a good dish.

### **`min_child_weight`**
- **What it does:** Sets the minimum total weight (sum of instance importance) of samples needed in a child (leaf).  
- **Why it matters:**  
  - Higher values = the model won’t create leaves that only explain very small groups → reduces overfitting.  
- **Think of it as…** setting a rule: “Don’t make a decision if only one or two people agree.” You need enough votes before splitting further.

### **`gamma`**
- **What it does:** Minimum loss reduction required to make a split.  
- **Why it matters:**  
  - Higher gamma makes the model more conservative, requiring a bigger gain to justify a new split.  
- **Think of it as…** raising the bar for making a new decision. Only big improvements are worth the effort.


## Dataset Setup
- Credit card fraud detection dataset
- 280k Transations with features
- Binary classification (Fraud/no fraud)
- Highly unbalanced (only 492 fraud examples)
- Features are V1-V28 (these are dementionally reduced features so we don't know what they actually represent)

In [ ]:
!ls ../input/creditcardfraud/ -l

In [ ]:
df = pd.read_csv("../input/creditcardfraud/creditcard.csv")
df.head()

In [ ]:
df["Class"].value_counts()

In [ ]:
X = df[[c for c in df.columns if c.startswith("V")] + ["Amount"]]
y = df["Class"]

# Basic Parameter Search

- We will focus only on the "max_depth" parameter.
- Pick 4 different values to check.
- For each parameter value we will run a 3-Fold cross validation and average the scores.
- At the end we can compare the scores for each value - we pick the hyperparameter value that scores the best!

In [ ]:
# Lets try to tune the "max_depth" parameter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

max_depths = [3, 5, 10, 15]

for max_depth in max_depths:
    # Train for each 3 fold cross validation
    # Print the average score at this parameter
    print(f"Training with max_depth {max_depth}")
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, va in skf.split(X.values, y.values):
        model = xgb.XGBClassifier(max_depth=max_depth)
        model.fit(X.values[tr], y.values[tr])
        preds = model.predict_proba(X.values[va])[:, 1]
        scores.append(roc_auc_score(y.values[va], preds))
    print(f"Max depth: {max_depth} - Average score {np.mean(scores)}")

# Advanced Parameter Tuning with Optuna

- The example above works fine for a single parameter
- In practice XGBoost has many parameters that we want to tune at once.
- We can use Optuna to help us explore a variety of different parameters.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
import optuna


# Defining the objective function
def objective(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }

    model = xgb.XGBClassifier(
        **param,
    )

    score = cross_val_score(model, X, y, cv=3, scoring="roc_auc").mean()
    return score

In [ ]:
# Create and run the optimization process with 100 trials
study = optuna.create_study(study_name="xgboost_study_cpu", direction="maximize")
study.optimize(objective, n_trials=10, show_progress_bar=True, n_jobs=-1)

# Retrieve the best parameter values
best_params = study.best_params
print(f"\nBest parameters: {best_params}")

In [ ]:
study.best_params

# Speed it up with GPU Support!

Hyperparameter tuning can take a lot of time to run - since we are training multiple models for each trial.

Any speed improvement will allow us to spend more time running experiments.

Since we have a GPU - all we need to do is change the classifier definition to see HUGE speedup:

```
model = xgb.XGBClassifier(device = 'cuda')  
```

In [ ]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
import optuna


# Defining the objective function
def objective_gpu(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }

    model = xgb.XGBClassifier(**param, device="cuda")

    score = cross_val_score(model, X, y, cv=3, scoring="roc_auc").mean()
    return score

In [ ]:
# Create and run the optimization process with 10 trials
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.optimize(objective_gpu, n_trials=10, show_progress_bar=True, n_jobs=-1)

# Retrieve the best parameter values
best_params = study.best_params
print(f"\nBest parameters: {best_params}")

In [ ]:
# Create and run the optimization process with 100 trials
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.optimize(objective_gpu, n_trials=10, show_progress_bar=True, n_jobs=-1)

# Retrieve the best parameter values
best_params = study.best_params
print(f"\nBest parameters: {best_params}")

# Making it even faster

- We've already sped up the hyperparameter search from 2mintes to ~25 seconds!

- We can use cupy to load our datasets to the GPU.
- This will speed up the inference time slightly
- We will need to manually write our cross validation look using `StratifiedKFold` to make this work.

In [ ]:
import cupy as cp
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X_gpu = cp.asarray(X.values)
y_gpu = cp.asarray(y.values)


def objective_gpu_cupy(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        # "tree_method": "hist",
        "device": "cuda",
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr, va in skf.split(
        cp.asnumpy(X_gpu), cp.asnumpy(y_gpu)
    ):  # indices on CPU are fine
        model = xgb.XGBClassifier(**param)
        model.fit(X_gpu[tr], y_gpu[tr])
        # GPU inference (no warning)
        p = model.predict_proba(X_gpu[va])[:, 1]
        # metric on CPU
        scores.append(roc_auc_score(cp.asnumpy(y_gpu[va]), cp.asnumpy(p)))
    return float(cp.asarray(scores).mean())

In [ ]:
# Create and run the optimization process with 100 trials
study = optuna.create_study(study_name="xgboost_study_gpu_cupy", direction="maximize")
study.optimize(objective_gpu_cupy, n_trials=100, show_progress_bar=True, n_jobs=-1)

# Retrieve the best parameter values
best_params = study.best_params
print(f"\nBest parameters: {best_params}")

# Big speedup!

- We were able to run 100 trials in ~3 minutes.
- On CPU we ran only 10 trials in 2 minutes!
- Expect a 5-15x speedup when running on GPU.

# Optuna Visualizations!

In [ ]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))